# CIFAR-10 Luokitteluprojekti: FCN vs CNN

## 1. Projektin tavoite

Projektin tavoitteena oli ratkaista CIFAR-10-kuvaluokitteluongelma kahdella eri neuroverkkoarkkitehtuurilla ja vertailla niiden toimintaa:

* Fully Connected Network (FCN)
* Convolutional Neural Network (CNN)

Vertailu tehtiin testitarkkuuden, oppimiskäyrien, parametrimäärän, opetusaikojen sekä virheellisten luokittelujen perusteella.

---

## 2. Data ja esikäsittely

CIFAR-10 koostuu 60 000 värikuvasta (32x32x3), joista 50 000 on koulutusdataa ja 10 000 testidataa. Luokkia on yhteensä 10.

Esikäsittelyvaiheet:

* Pikseliarvot skaalattiin välille [0,1]
* Luokat muunnettiin one-hot-muotoon

Toteutus tehtiin Keras 3:lla TensorFlow-backendilla.

---

## 3. Mallit

### FCN

* Flatten
* Dense(512, relu) + BatchNormalization + Dropout(0.4)
* Dense(256, relu) + BatchNormalization + Dropout(0.4)
* Dense(10, softmax)
* Optimointi: Adam (learning rate 0.0005)
* Batch size: 128
* Epochit: 20

### CNN

* Data Augmentation: RandomFlip, RandomRotation (0.1), RandomZoom (0.1)
* Conv2D(32, 3x3, relu) + 2x BatchNormalization + MaxPooling2D + Dropout(0.3)
* Conv2D(64, 3x3, relu) + 2x BatchNormalization + MaxPooling2D + Dropout(0.3)
* Conv2D(128, 3x3, relu) + 2x BatchNormalization + MaxPooling2D + Dropout(0.3)
* Flatten + Dense(256, relu) + BatchNormalization + Dropout(0.5)
* Dense(10, softmax)
* Optimointi: Adam (learning rate 0.0005)
* Batch size: 128
* Epochit: 20 (EarlyStopping käytössä)

---

## 4. Mitatut tulokset

Molemmat mallit koulutettiin samalla koneella samalla datalla. Opetusaika mitattiin koko fit-vaiheesta.

| Malli | Testitarkkuus | Testiloss | Parametreja |          Opetusaika |
| ----- | ------------: | --------: | ----------: | ------------------: |
| FCN   |       46.41 % |    1.4939 |   1 710 346 | 491.31 s (8.19 min) |
| CNN   |       73.66 % |    0.7624 |     816 938 | 1890 s   (31.5 min) |

Tavoitteisiin peilattuna:

* FCN tavoite 50–55 % (realistinen): toteutunut tarkkuus 46.41 % jäi tavoitteen alle.
* CNN tavoite 75–80 % (realistinen): toteutunut tarkkuus 73.66 % pääsi erittäin lähelle tavoitetta.

Mahdollisia syitä tähän:

* Mallit olivat melko yksinkertaisia CIFAR-10-tehtävään
* Hyperparametreja ei optimoitu systemaattisesti
* CNN-mallissa käytettiin voimakasta regularisointia (Data Augmentation ja paljon Dropoutia), minkä vuoksi malli oppii hitaammin. Se olisi todennäköisesti saavuttanut 80 % rajan, jos epookkien määrää olisi nostettu esimerkiksi sataan asti.

---

## 5. Oppimiskäyrien analyysi

### FCN

* Viimeinen train accuracy: 44.20 %
* Viimeinen validation accuracy: 46.86 %
* Viimeinen validation loss: 1.5088

FCN-mallissa train- ja validation-käyrät pysyivät melko lähellä toisiaan, mutta taso jäi matalaksi. Tämä viittaa enemmän alisovittamiseen (underfitting) kuin ylisovittamiseen.

### CNN

* Viimeinen train accuracy: 73.93 %
* Viimeinen validation accuracy: 73.09 %
* Viimeinen validation loss: 0.8022

CNN oppimiskäyrät olivat erinomaiset. Opetusdatan ja validointidatan tarkkuudet kulkivat rinta rinnan läpi koko koulutuksen. Tämä osoittaa, että Data Augmentation ja raskaat Dropout-kerrokset estivät ylisovittumisen (overfitting) lähes täydellisesti. Malli yleistää oppimansa hienosti uuteen dataan.

---

## 6. Miksi CNN toimi paremmin kuin FCN

Tulokset tukevat teoriaa: CNN soveltuu kuvadatalle paremmin kuin FCN.

Keskeiset syyt:

* Konvoluutiokerrokset oppivat paikallisia piirteitä (reunat, tekstuurit, muodot)
* CNN säilyttää kuvien spatiaalisen rakenteen
* Painojen jakaminen (weight sharing) vähentää parametrien määrää ja parantaa yleistymistä
* FCN litistää kuvan vektoriksi, jolloin tärkeä rakenne katoaa

Huomionarvoista on, että CNN saavutti paremman tarkkuuden, vaikka sillä oli selvästi vähemmän parametreja kuin FCN:llä.

---

## 7. Visualisoinnit ja virheelliset luokittelut

Notebookeissa tarkasteltiin:

* Ennusteita testikuville
* Luokkien todennäköisyysjakaumia (softmax)
* Väärin luokiteltuja kuvia

Havaintoja:

* FCN teki enemmän virheitä samankaltaisten luokkien välillä (esim. cat vs dog)
* CNN tunnisti piirteitä paremmin
* Virheitä tuli erityisesti tilanteissa, joissa:

  * kohde oli pieni
  * kohde oli osittain peittynyt
  * tausta oli hallitseva

---

## 8. Johtopäätökset

Projektin perusteella CNN oli selkeästi FCN:ää parempi CIFAR-10-tehtävässä:

* korkeampi testitarkkuus (+27.25 prosenttiyksikköä)
* pienempi testiloss
* vähemmän parametreja
* pidempi opetusaika (n. 31 min vs 8 min), mikä on odotettavaa CNN-mallin syvemmän rakenteen ja reaaliaikaisen data-augmentaation vuoksi. Suorituskyvyn merkittävä parannus kuitenkin oikeuttaa tämän.

Vaikka kumpikaan malli ei yltänyt realistiseen tavoitetasoon, CNN antoi selvästi paremman lähtötason jatkokehitykselle.

---

## 9. Jatkokehitysideat

Tarkkuutta voisi parantaa seuraavilla keinoilla:

* Laajempi data augmentation (esim. random crop ja color jitter nykyisten lisäksi)
* Learning rate scheduler (esim. ReduceLROnPlateau)
* Syvempi CNN-arkkitehtuuri
* L2-regularisointi ja parempi dropout-säätö
* Mallin tallennus parhaassa epochissa (checkpoint)

Näillä keinoilla olisi mahdollista päästä noin 80–85 % tarkkuuteen.

---

## 10. Mitä opittiin

Projektin aikana opittiin:

* miten eri neuroverkkoarkkitehtuurit vaikuttavat suorituskykyyn
* miksi CNN on parempi kuvadatalle kuin FCN
* miten overfitting ja underfitting tunnistetaan
* miten mallien suorituskykyä analysoidaan käytännössä

Lisäksi opittiin, että suurempi parametrimäärä ei automaattisesti tarkoita parempaa mallia.
